# Mammalcam: fine-tuning the species classifier on a GPU

Runs on **Google Colab** (Runtime → Change runtime type → T4 GPU) or **Kaggle** (Settings → Accelerator: GPU, Internet: on).

What it does:

1. clones the repository and installs it
2. downloads CCT20 (6.5 GB) and LILA's MegaDetector results (5 MB), see the README for sources and licences
3. rebuilds the camera splits and the MegaDetector crops exactly as in the repository
4. fine-tunes `convnext_tiny.fb_in22k` on the crops with the **same 5-fold camera cross-validation** as the linear probe, and predicts every test image end to end
5. packs the predictions for download

Expected time on a T4: about 10 minutes for data and crops, then roughly 15–25 minutes per fold.
Each finished fold is saved immediately; if the session ends, run the notebook again and it continues with the next fold
(on Colab, put the output on Google Drive so it survives the session).

No keys or credentials are needed.

In [ ]:
import os, subprocess, sys
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else os.getcwd()
print("Colab" if ON_COLAB else "Kaggle" if ON_KAGGLE else "local", "| base:", BASE)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Repository

The notebook uses the code as pushed to GitHub, so push the latest commits first.

In [ ]:
REPO = "https://github.com/hannes-vnz/Mammalcam.git"
ROOT = os.path.join(BASE, "Mammalcam")
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
!git log --oneline -1

In [ ]:
# torch, torchvision, pandas and scikit-learn are preinstalled on Colab and Kaggle; don't reinstall torch
!pip install -q -e . --no-deps
!pip install -q "timm>=1.0" "tabulate>=0.9"
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| CUDA", torch.cuda.is_available())

## 2. Where to keep the results

On Colab, mount Google Drive so finished folds survive a disconnect. On Kaggle, `/kaggle/working` is kept as notebook output.

In [ ]:
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/mammalcam/finetune"
else:
    OUT = os.path.join(ROOT, "outputs", "finetune")
os.makedirs(OUT, exist_ok=True)
print("results go to", OUT)

## 3. Data, splits and crops

Same scripts and seeds as locally. The split step must reproduce the committed `configs/splits_cct20.json`
(the check below fails otherwise), and the fine-tuning script checks the camera groups against
`reports/results/classify.json`.

In [ ]:
!python -m mammalcam.data.download cct20
!python -m mammalcam.data.download cct-md

In [ ]:
!python -m mammalcam.data.split
!git diff --exit-code configs/splits_cct20.json && echo "splits identical to the repository"

In [ ]:
!python -m mammalcam.crops.make_crops --workers {os.cpu_count()}

## 4. Smoke test (about a minute)

A few steps on a small subset, written to a separate folder so it never counts as a finished fold.

In [ ]:
!python -m mammalcam.models.finetune --folds 0 --limit 256 --epochs 1 --max-steps 5 --out outputs/smoke

## 5. Fine-tuning with 5-fold camera cross-validation

Resumable: folds whose `fold_<k>.csv` already exists in `OUT` are skipped. To spread the work over several
sessions, run a subset, e.g. `--folds 0,1`.

In [ ]:
WORKERS = min(8, os.cpu_count())
!python -m mammalcam.models.finetune --epochs 6 --batch-size 64 --lr 1e-4 --workers {WORKERS} --out {OUT}

## 6. Quick look

In [ ]:
import json, glob
import pandas as pd
pred = pd.read_csv(os.path.join(OUT, "predictions.csv"))
summary = pred.assign(image=pred.crops_image == pred.label, burst=pred.crops_sequence == pred.label) \
    .groupby("evaluation")[["image", "burst"]].mean()
print(summary.round(3))
print("pooled accuracy: per image %.3f, with burst average %.3f" % (
    (pred.crops_image == pred.label).mean(), (pred.crops_sequence == pred.label).mean()))
for path in sorted(glob.glob(os.path.join(OUT, "fold_*_history.json"))):
    print(os.path.basename(path), [round(h["val_macro_f1"], 3) for h in json.load(open(path))])

## 7. Download

Copy `predictions.csv` into the local repository as `data/predictions/finetune_test.csv`, then run
`python -m mammalcam.eval.compare` to put the fine-tuned model next to the linear probe and SpeciesNet.
The `fold_*_history.json` files hold the training curves.

In [ ]:
import shutil
archive = shutil.make_archive(os.path.join(BASE, "mammalcam_finetune"), "zip", OUT)
print("packed", archive)
if ON_COLAB:
    from google.colab import files
    files.download(archive)